# Universal Data Analysis and Exploratory Data Analysis System

## Project Title

### Generic Data Analysis and Automated EDA Reporting System

---

## Objective

Develop a reusable, domain-independent Python framework capable of automatically analyzing any structured CSV dataset by performing data validation, cleaning, profiling, exploratory data analysis, visualization, statistical analysis, automated reporting, and export of analytical outputs.

The framework is designed so that only the input dataset changes while the analysis workflow remains reusable across different domains.

## 1.Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

from scipy import stats
from ydata_profiling import ProfileReport

## 2.Load dataset

In [ ]:
import os
import pandas as pd

def load_dataset():
   

    try:
        file_path = input("Enter CSV file path: ").strip().strip('"')

        if not file_path:
            raise ValueError("No file path entered.")

        if not os.path.isfile(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")

        if not file_path.lower().endswith(".csv"):
            raise ValueError("Please provide a CSV file.")

        df = pd.read_csv(file_path)

        print(" Dataset loaded successfully.")
        print(f"Shape: {df.shape}")

        return df

    except FileNotFoundError as e:
        print(f" {e}")

    except ValueError as e:
        print(f" {e}")

    except pd.errors.EmptyDataError:
        print(" The CSV file is empty.")

    except pd.errors.ParserError:
        print(" Unable to parse the CSV file.")

    except Exception as e:
        print(f" Unexpected Error: {e}")

    return None


df = load_dataset()

if df is None:
    raise SystemExit("Dataset loading failed. Please provide a valid CSV file.")

## 3.data profiling

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

In [ ]:
df.describe

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
df.nunique()

In [ ]:
df.memory_usage(deep=True)

In [ ]:
print("=" * 50)
print("DATA PROFILING SUMMARY")
print("=" * 50)

print(f"Shape: {df.shape}")
print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

In [ ]:
print("=" * 50)
print("DATA PROFILING SUMMARY")
print("=" * 50)

print(f"Shape: {df.shape}")
print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nNumeric Columns:")
print(df.select_dtypes(include="number").columns.tolist())

print("\nCategorical Columns:")
print(df.select_dtypes(exclude="number").columns.tolist())

print("\nDataset Memory Usage (MB):")
print(round(df.memory_usage(deep=True).sum()/1024**2,2))


### 4.Data Cleaning & Preprocessing

In [ ]:
print("Duplicate Rows Before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate Rows After:", df.duplicated().sum())

In [ ]:
print("New Shape:", df.shape)

In [ ]:
df.to_csv("cleaned_data/student_performance_cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

## Outlier Detection and Handling

Outlier detection identifies extreme observations that may influence statistical analysis and visualization.

This framework applies the **Interquartile Range (IQR)** method to each numerical column. Since the project is designed as a reusable analysis system, outliers are identified and reported rather than automatically removed.

In [ ]:
print("=" * 60)
print("OUTLIER DETECTION USING IQR")
print("=" * 60)

numerical_columns = df.select_dtypes(include="number").columns

outlier_summary = {}

for col in numerical_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    outlier_summary[col] = len(outliers)

outlier_df = pd.DataFrame(
    outlier_summary.items(),
    columns=["Column", "Number of Outliers"]
)

print(outlier_df)

outlier_df.to_csv(
    "cleaned_data/outlier_summary.csv",
    index=False
)

## Feature Engineering

Feature engineering creates new informative variables from existing features.

Since this notebook is designed to work with any structured dataset, only generic features are created automatically whenever suitable numeric columns are available.

In [ ]:
df = df.copy()

numeric_cols = df.select_dtypes(include="number").columns

if len(numeric_cols) >= 2:

    df.loc[:, "Average_Numeric_Value"] = df[numeric_cols].mean(axis=1)

    print("New Feature Created Successfully!")
    print(df[["Average_Numeric_Value"]].head())

else:

    print("Not enough numeric columns for feature engineering.")

In [ ]:
import os

print(os.getcwd())

In [ ]:
import os

print(os.listdir())

In [ ]:
import os

print(os.path.isdir("cleaned_data"))
print(os.path.isfile("cleaned_data"))

In [ ]:
import os

print("Current directory:", os.getcwd())
print("Folder exists:", os.path.exists("cleaned_data"))

df.to_csv("./cleaned_data/student_performance_cleaned.csv", index=False)

print(" Dataset saved successfully!")

In [ ]:
import os

print(os.listdir("cleaned_data"))

In [ ]:
df_saved = pd.read_csv("cleaned_data/student_performance_cleaned.csv")

df_saved.head()

In [ ]:
print("Original cleaned DataFrame:", df.shape)
print("Saved CSV:", df_saved.shape)

## 5.Univarite Analysis

In [ ]:
df.describe().T

In [ ]:
numerical_columns = df.select_dtypes(include=["number"]).columns.tolist()

categorical_columns = df.select_dtypes(exclude=["number"]).columns.tolist()

## Histogram

In [ ]:
for column in numerical_columns:

    plt.figure(figsize=(6,4))

    sns.histplot(df[column], kde=True)

    plt.title(column)

    plt.show()

## Boxplots

In [ ]:
import math
import matplotlib.pyplot as plt
import seaborn as sns

num_cols = len(numerical_columns)
cols = 3
rows = math.ceil(num_cols / cols)

plt.figure(figsize=(15, 5 * rows))

for i, column in enumerate(numerical_columns, 1):
    plt.subplot(rows, cols, i)
    sns.boxplot(y=df[column])
    plt.title(column)

plt.tight_layout()
plt.show()

## Count Plots

In [ ]:
if len(categorical_columns) == 0:

    print("No categorical columns available.")

else:

    for column in categorical_columns:

        if df[column].nunique() > 20:
            continue

        plt.figure(figsize=(6,4))

        sns.countplot(
            data=df,
            x=column
        )

        plt.title(f"Distribution of {column}")

        plt.xticks(rotation=45)

        plt.tight_layout()

        plt.show()

In [ ]:
print(df.dtypes)

In [ ]:
print(df.select_dtypes(include=["object", "category"]).columns.tolist())

# 6.Bivariate Analysis

In [ ]:
correlation = df[numerical_columns].corr()

correlation

# Correlation Heatmap

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Correlation Heatmap")

plt.show()

## Correlation Analysis Interpretation

The correlation matrix helps identify linear relationships between numerical variables.

- Positive values indicate a direct relationship.
- Negative values indicate an inverse relationship.
- Values close to zero indicate weak or no linear relationship.

This analysis assists in identifying patterns, redundancy, and feature importance before predictive modeling.

# Scatter Plot

In [ ]:
from itertools import combinations

if len(numerical_columns) < 2:

    print("Not enough numerical columns for regression plots.")

else:

    for x_col, y_col in combinations(numerical_columns, 2):

        plt.figure(figsize=(8,6))

        sns.regplot(
            data=df,
            x=x_col,
            y=y_col,
            scatter_kws={"alpha":0.5}
        )

        plt.title(f"{x_col} vs {y_col}")

        plt.tight_layout()

        plt.show()

In [ ]:
if len(numerical_columns) >= 3:
    plt.figure(figsize=(8,6))
    sns.regplot(data=df,
                x=numerical_columns[0],
                y=numerical_columns[2],
                scatter_kws={"alpha":0.5})
    plt.title(f"{numerical_columns[0]} vs {numerical_columns[2]}")
    plt.show()

In [ ]:
if len(numerical_columns) >= 4:
    plt.figure(figsize=(8,6))
    sns.regplot(data=df,
                x=numerical_columns[1],
                y=numerical_columns[3],
                scatter_kws={"alpha":0.5})
    plt.title(f"{numerical_columns[1]} vs {numerical_columns[3]}")
    plt.show()

# Grouped Bar Chart

In [ ]:
if len(categorical_columns) == 0 or len(numerical_columns) == 0:

    print("Categorical and numerical columns are required for grouped bar charts.")

else:

    for cat_col in categorical_columns:

        if df[cat_col].nunique() > 20:
            continue

        for num_col in numerical_columns:

            plt.figure(figsize=(8,6))

            sns.barplot(
                data=df,
                x=cat_col,
                y=num_col
            )

            plt.title(f"Average {num_col} by {cat_col}")

            plt.xticks(rotation=45)

            plt.tight_layout()

            plt.show()

In [ ]:
if len(categorical_columns) == 0 or len(numerical_columns) == 0:

    print("Categorical and numerical columns are required.")

else:

    for cat_col in categorical_columns:

        if df[cat_col].nunique() > 20:
            continue

        for num_col in numerical_columns:

            plt.figure(figsize=(7,5))

            sns.barplot(
                data=df,
                x=cat_col,
                y=num_col,
                estimator="mean",
                errorbar=None
            )

            plt.title(f"Average {num_col} by {cat_col}")

            plt.xticks(rotation=45)

            plt.tight_layout()

            plt.show()

# Boxplot

In [ ]:
if len(categorical_columns) == 0 or len(numerical_columns) == 0:

    print("Need at least one categorical and one numerical column.")

else:

    for cat_col in categorical_columns:

        if df[cat_col].nunique() > 20:
            continue

        for num_col in numerical_columns:

            plt.figure(figsize=(8,6))

            sns.boxplot(
                data=df,
                x=cat_col,
                y=num_col
            )

            plt.title(f"{num_col} by {cat_col}")

            plt.xticks(rotation=45)

            plt.tight_layout()

            plt.show()

In [ ]:
if len(numerical_columns) >= 2:

    plt.figure(figsize=(8,6))

    sns.regplot(
        data=df,
        x=numerical_columns[1],
        y=numerical_columns[2],
        scatter_kws={"alpha":0.6}
    )

    plt.title(f"{numerical_columns[0]} vs {numerical_columns[1]}")
    plt.tight_layout()
    plt.show()

else:
    print("Need at least two numerical columns.")

## Multivariate Analysis

Multivariate analysis examines relationships among multiple numerical variables simultaneously.

A pair plot provides scatter plots between numerical features along with their distributions, helping identify clusters, trends, and potential correlations.

In [ ]:
if len(numerical_columns) < 2:

    print("Not enough numerical columns available.")

else:

    selected_cols = numerical_columns[:5]

    pair_grid = sns.pairplot(df[selected_cols])

    pair_grid.figure.suptitle(
        "Pairwise Relationship Analysis",
        y=1.02,
        fontsize=16
    )

    plt.show()

## Automatic Scatter Plot Generation

The framework automatically generates scatter plots between numerical variables and the selected target variable to identify relationships, trends and possible outliers.

In [ ]:
from itertools import combinations

if len(numerical_columns) < 2:

    print("Need at least two numerical columns.")

else:

    for x_col, y_col in combinations(numerical_columns, 2):

        plt.figure(figsize=(7,5))

        sns.regplot(
            data=df,
            x=x_col,
            y=y_col,
            scatter_kws={"alpha":0.5},
            line_kws={"color":"red"}
        )

        plt.title(f"{y_col} vs {x_col}")

        plt.tight_layout()

        plt.show()

## Automatic Bar Chart Generation

Automatically compares the average numerical values across categorical features.

In [ ]:
if len(categorical_columns) == 0 or len(numerical_columns) == 0:

    print("Need at least one categorical and one numerical column.")

else:

    for cat_col in categorical_columns:

        if df[cat_col].nunique() > 20:
            continue

        for num_col in numerical_columns:

            plt.figure(figsize=(6,4))

            sns.barplot(
                data=df,
                x=cat_col,
                y=num_col,
                estimator="mean",
                errorbar=None
            )

            plt.title(f"Average {num_col} by {cat_col}")

            plt.xticks(rotation=45)

            plt.tight_layout()

            plt.show()

In [ ]:
plt.figure(figsize=(8,6))

correlation = df[numerical_columns].corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
if len(numerical_columns) >= 2:

    selected_columns = numerical_columns[:4]  # Use up to the first 4 numerical columns

    sns.pairplot(df[selected_columns])

    plt.show()

else:
    print("Need at least two numerical columns for a pairplot.")

## Observation

- The selected numerical variables show varying degrees of positive or negative relationships.
- Some numerical variables appear to be positively correlated, while others show weak or no correlation.
- The strength and direction of these relationships depend on the dataset being analyzed.

In [ ]:
categorical_columns = []
numerical_columns = []

for col in df.columns:

    if df[col].nunique() <= 1:
        continue

    if (
        pd.api.types.is_object_dtype(df[col])
        or pd.api.types.is_categorical_dtype(df[col])
    ):

        categorical_columns.append(col)

    elif pd.api.types.is_numeric_dtype(df[col]):

        if df[col].nunique() <= 10:
            categorical_columns.append(col)
        else:
            numerical_columns.append(col)

print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

In [ ]:
categorical_columns = []
numerical_columns = []

for col in df.columns:

    if df[col].nunique() <= 1:
        continue

    if pd.api.types.is_numeric_dtype(df[col]):

        if df[col].nunique() <= 10:
            categorical_columns.append(col)
        else:
            numerical_columns.append(col)

    else:
        categorical_columns.append(col)

In [ ]:
from itertools import combinations

if len(categorical_columns) < 2 or len(numerical_columns) == 0:

    print("Need at least two categorical and one numerical column.")

else:

    for num_col in numerical_columns:

        for cat1, cat2 in combinations(categorical_columns, 2):

            # Skip high-cardinality categorical columns
            if df[cat1].nunique() > 15 or df[cat2].nunique() > 15:
                continue

            pivot = pd.pivot_table(
                data=df,
                values=num_col,
                index=cat1,
                columns=cat2,
                aggfunc="mean"
            )

            if pivot.empty:
                continue

            plt.figure(figsize=(8,6))

            sns.heatmap(
                pivot,
                annot=True,
                cmap="YlGnBu",
                fmt=".2f"
            )

            plt.title(f"Average {num_col} by {cat1} and {cat2}")

            plt.tight_layout()

            plt.show()

In [ ]:
from itertools import combinations

if len(categorical_columns) < 2 or len(numerical_columns) == 0:

    print("Need at least two categorical and one numerical column.")

else:

    for num_col in numerical_columns:

        for cat1, cat2 in combinations(categorical_columns, 2):

            if df[cat1].nunique() > 15 or df[cat2].nunique() > 15:
                continue

            pivot = pd.pivot_table(
                data=df,
                values=num_col,
                index=cat1,
                columns=cat2,
                aggfunc="mean"
            )

            if pivot.empty:
                continue

            print(f"\nPivot Table: {num_col} by {cat1} and {cat2}")
            print(pivot)

In [ ]:
from itertools import combinations

if len(categorical_columns) < 2 or len(numerical_columns) == 0:

    print("Need at least two categorical and one numerical column.")

else:

    for num_col in numerical_columns:

        for cat1, cat2 in combinations(categorical_columns, 2):

            if df[cat1].nunique() > 15 or df[cat2].nunique() > 15:
                continue

            pivot = pd.pivot_table(
                data=df,
                values=num_col,
                index=cat1,
                columns=cat2,
                aggfunc="mean"
            )

            if pivot.empty:
                continue

            plt.figure(figsize=(6,5))

            sns.heatmap(
                pivot,
                annot=True,
                cmap="YlGnBu",
                fmt=".2f"
            )

            plt.title(f"Average {num_col} by {cat1} and {cat2}")

            plt.tight_layout()

            plt.show()

In [ ]:
from itertools import combinations

if len(categorical_columns) < 2 or len(numerical_columns) == 0:

    print("Need at least two categorical and one numerical column.")

else:

    for num_col in numerical_columns:

        for x_col, hue_col in combinations(categorical_columns, 2):

            # Skip high-cardinality categorical columns
            if df[x_col].nunique() > 15 or df[hue_col].nunique() > 15:
                continue

            plt.figure(figsize=(8,6))

            sns.boxplot(
                data=df,
                x=x_col,
                y=num_col,
                hue=hue_col
            )

            plt.title(f"{num_col} by {x_col} and {hue_col}")

            plt.xticks(rotation=45)

            plt.tight_layout()

            plt.show()

In [ ]:
if len(categorical_columns) == 0 or len(numerical_columns) == 0:

    print("Need at least one categorical and one numerical column.")

else:

    for cat_col in categorical_columns:

        if df[cat_col].nunique() > 20:
            continue

        for num_col in numerical_columns:

            plt.figure(figsize=(8,6))

            sns.violinplot(
                data=df,
                x=cat_col,
                y=num_col
            )

            plt.title(f"Distribution of {num_col} by {cat_col}")

            plt.xticks(rotation=45)

            plt.tight_layout()

            plt.show()

In [ ]:
from itertools import combinations

if len(categorical_columns) < 2:

    print("Need at least two categorical columns.")

else:

    for col1, col2 in combinations(categorical_columns, 2):

        # Skip high-cardinality columns
        if df[col1].nunique() > 20 or df[col2].nunique() > 20:
            continue

        cross = pd.crosstab(
            df[col1],
            df[col2]
        )

        print(f"\nCrosstab: {col1} vs {col2}")
        print(cross)

In [ ]:
from itertools import combinations

if len(categorical_columns) < 2:

    print("Need at least two categorical columns.")

else:

    for col1, col2 in combinations(categorical_columns, 2):

        # Skip high-cardinality columns
        if df[col1].nunique() > 20 or df[col2].nunique() > 20:
            continue

        cross = pd.crosstab(
            df[col1],
            df[col2]
        )

        print(f"\nCrosstab: {col1} vs {col2}")
        print(cross)

## Observations

- Numerical variables exhibit different distributions and may contain outliers.
- The box plot helps identify the spread and potential outliers in the selected numerical variable.
- Scatter plots reveal the strength and direction of relationships between numerical variables.
- Grouped visualizations show how numerical values vary across different categorical groups.
- Correlation analysis helps identify positive, negative, or weak relationships among numerical variables.

# 8. Statistical Analysis

In [ ]:
from scipy import stats

In [ ]:
if len(numerical_columns) >= 2:

    correlation, p_value = stats.pearsonr(
        df[numerical_columns[0]],
        df[numerical_columns[1]]
    )

    print(
        f"Correlation between {numerical_columns[0]} and {numerical_columns[1]}:",
        correlation
    )
    print("P-value:", p_value)

else:
    print("Need at least two numerical columns for Pearson correlation.")

In [ ]:
if len(categorical_columns) >= 1 and len(numerical_columns) >= 1:

    groups = df[categorical_columns[0]].dropna().unique()

    if len(groups) == 2:

        group1 = df[df[categorical_columns[0]] == groups[0]][numerical_columns[0]]
        group2 = df[df[categorical_columns[0]] == groups[1]][numerical_columns[0]]

        t_stat, p_value = stats.ttest_ind(group1, group2)

        print("T-statistic:", t_stat)
        print("P-value:", p_value)

    else:
        print(
            f"{categorical_columns[0]} must have exactly two groups for an independent t-test."
        )

else:
    print("Need at least one categorical column and one numerical column.")

In [ ]:
if len(categorical_columns) >= 1 and len(numerical_columns) >= 1:

    groups = [
        group[numerical_columns[0]].dropna().values
        for _, group in df.groupby(categorical_columns[0])
    ]

    if len(groups) >= 2:

        f_stat, p_value = stats.f_oneway(*groups)

        print("ANOVA Test")
        print("F-statistic:", f_stat)
        print("P-value:", p_value)

    else:

        print("Need at least two groups for ANOVA.")

else:

    print("Suitable columns not found.")

In [ ]:
if len(categorical_columns) >= 2:

    contingency_table = pd.crosstab(
        df[categorical_columns[0]],
        df[categorical_columns[1]]
    )

    chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

    print("Chi-square:", chi2)
    print("Degrees of Freedom:", dof)
    print("P-value:", p_value)

else:
    print("Need at least two categorical columns for the Chi-square test.")

## Summary Table

Create this table after running all the statistical tests and replace the **Result** column based on your outputs.

| Test | Variables | Result |
|------|-----------|--------|
| Pearson Correlation | `numerical_columns[0]` vs `numerical_columns[1]` | Positive/Negative, Significant/Not Significant |
| Independent T-Test | `categorical_columns[0]` vs `numerical_columns[0]` | Significant/Not Significant |
| One-Way ANOVA | `categorical_columns[1]` vs `numerical_columns[1]` | Significant/Not Significant |
| Chi-Square Test | `categorical_columns[0]` vs `categorical_columns[1]` | Significant/Not Significant |

# 9. Visualization Dashboard

In [ ]:
plt.figure(figsize=(18, 12))

# 1. Histogram
if len(numerical_columns) >= 1:
    plt.subplot(2, 2, 1)
    sns.histplot(df[numerical_columns[0]], bins=20, kde=True)
    plt.title(f"Distribution of {numerical_columns[0]}")

# 2. Box Plot
if len(numerical_columns) >= 1:
    plt.subplot(2, 2, 2)
    sns.boxplot(y=df[numerical_columns[0]])
    plt.title(f"Box Plot of {numerical_columns[0]}")

# 3. Scatter Plot
if len(numerical_columns) >= 2:
    plt.subplot(2, 2, 3)
    sns.regplot(
        data=df,
        x=numerical_columns[0],
        y=numerical_columns[1],
        scatter_kws={"alpha": 0.5}
    )
    plt.title(f"{numerical_columns[0]} vs {numerical_columns[1]}")

# 4. Bar Chart
if len(categorical_columns) >= 1 and len(numerical_columns) >= 2:
    plt.subplot(2, 2, 4)
    sns.barplot(
        data=df,
        x=categorical_columns[0],
        y=numerical_columns[1]
    )
    plt.title(f"Average {numerical_columns[1]} by {categorical_columns[0]}")
    plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig("reports/dashboard.png", dpi=300)

plt.show()

In [ ]:
plt.figure(figsize=(15,10))

# 1. Correlation Heatmap
if len(numerical_columns) >= 2:
    plt.subplot(2,2,1)
    sns.heatmap(
        df[numerical_columns].corr(),
        annot=True,
        cmap="coolwarm"
    )
    plt.title("Correlation Heatmap")

# 2. Box Plot
if len(numerical_columns) >= 1:
    plt.subplot(2,2,2)
    sns.boxplot(
        data=df,
        y=numerical_columns[0]
    )
    plt.title(f"Box Plot of {numerical_columns[0]}")

# 3. Histogram
if len(numerical_columns) >= 2:
    plt.subplot(2,2,3)
    sns.histplot(
        df[numerical_columns[1]],
        bins=15
    )
    plt.title(f"Distribution of {numerical_columns[1]}")

# 4. Scatter Plot
if len(numerical_columns) >= 3:
    plt.subplot(2,2,4)
    sns.scatterplot(
        data=df,
        x=numerical_columns[2],
        y=numerical_columns[0]
    )
    plt.title(f"{numerical_columns[2]} vs {numerical_columns[0]}")

plt.tight_layout()
plt.show()

# 10.Automated EDA Report

In [ ]:
try:
    from ydata_profiling import ProfileReport
except ImportError:
    import sys
    !{sys.executable} -m pip install ydata-profiling
    from ydata_profiling import ProfileReport

In [ ]:
import os

os.makedirs("reports", exist_ok=True)

In [ ]:
profile = ProfileReport(
    df,
    title="Generic Data Analysis Report",
    explorative=True
)

In [ ]:
profile.to_file("reports/EDA_Report.html")

print("EDA Report generated successfully!")

In [ ]:
os.makedirs("cleaned_data", exist_ok=True)

df.to_csv(
    "cleaned_data/cleaned_dataset.csv",
    index=False
)

print("Cleaned Dataset Exported Successfully!")

In [ ]:
summary = df.describe(include="all")

summary.to_csv(
    "reports/summary_statistics.csv"
)

print("Summary Statistics Exported!")

In [ ]:
if len(numerical_columns) > 1:

    correlation = df[numerical_columns].corr()

    correlation.to_csv(
        "reports/correlation_matrix.csv"
    )

    print("Correlation Matrix Exported!")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.savefig(
    "reports/dashboard.png",
    dpi=300,
    bbox_inches="tight"
)

print("Dashboard Saved Successfully!")

# 11. Business Insights & Recommendations

Based on the exploratory data analysis and statistical findings, the following observations and recommendations can help improve data quality and support better business decisions.

In [ ]:
print("========== DATASET SUMMARY ==========")

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nMissing Values")
print(df.isnull().sum().sum())

print("\nDuplicate Records")
print(df.duplicated().sum())

print("\nNumerical Columns")
print(len(numerical_columns))

print("\nCategorical Columns")
print(len(categorical_columns))

print("\nDataset Memory Usage:")
print(df.memory_usage(deep=True).sum() / 1024, "KB")

print("\nTotal Numeric Columns:", len(numerical_columns))
print("Total Categorical Columns:", len(categorical_columns))

## Recommendations
- Monitor data quality regularly.
- Validate incoming data before analysis.
- Standardize categorical values.
- Automate data cleaning for recurring datasets.
- Review highly correlated features before predictive modeling.

# 12. Conclusion

This project successfully developed a reusable and domain-independent data analysis framework capable of analyzing structured CSV datasets. The system performs data loading, profiling, cleaning, visualization, statistical analysis, dashboard generation, automated EDA report generation, and export of analytical results.

The framework can be reused with datasets from different domains with minimal code changes.

# 13. Future Improvements

- Support Excel (.xlsx) and JSON datasets.
- Build an interactive dashboard using Streamlit or Plotly Dash.
- Add machine learning models for prediction.
- Generate PDF reports.
- Connect directly to SQL databases.
- Automatically detect target variables.

In [ ]:
import os

print("Reports folder exists:", os.path.exists("reports"))
print("EDA Report exists:", os.path.exists("reports/EDA_Report.html"))

In [ ]:
import os

print("Current Folder:", os.getcwd())

print("\nFiles in reports folder:")
if os.path.exists("reports"):
    print(os.listdir("reports"))
else:
    print("reports folder not found")

print("\nFiles in cleaned_data folder:")
if os.path.exists("cleaned_data"):
    print(os.listdir("cleaned_data"))
else:
    print("cleaned_data folder not found")

In [ ]:
from IPython.display import IFrame

IFrame(src="reports/EDA_Report.html", width=1000, height=600)